In [ ]:
#|hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai  # upgrade fastai on colab

In [ ]:
#|export
from __future__ import annotations
from fastai.basics import *
from fastai.tabular.core import *
from fastai.tabular.model import *
from fastai.tabular.data import *

In [ ]:
#|hide
from nbdev.showdoc import *

In [ ]:
#|default_exp tabular.learner

# Tabular learner

> The function to immediately get a `Learner` ready to train for tabular data

The main function you probably want to use in this module is `tabular_learner`. It will automatically create a `TabularModel` suitable for your data and infer the right loss function. See the [tabular tutorial](http://docs.fast.ai/tutorial.tabular.html) for an example of use in context.

## Main functions

In [ ]:
#|export
class TabularLearner(Learner):
    "`Learner` for tabular data"
    def predict(self, 
        row:pd.Series, # Features to be predicted
    ):
        "Predict on a single sample"
        dl = self.dls.test_dl(row.to_frame().T)
        dl.dataset.conts = dl.dataset.conts.astype(np.float32)
        inp,preds,_,dec_preds = self.get_preds(dl=dl, with_input=True, with_decoded=True)
        b = (*tuplify(inp),*tuplify(dec_preds))
        full_dec = self.dls.decode(b)
        return full_dec,dec_preds[0],preds[0]

    def counterfactual(self,
        row:pd.Series, # A single sample (same format as `predict` expects)
        target_class, # Desired class: an int index or a label in `self.dls.vocab`
        cont_step:float=0.1, # Step size for continuous nudges, scaled by each feature's raw std
        max_iter:int=100, # Maximum number of greedy perturbation steps
    ):
        "Greedily perturb one feature at a time to flip the prediction of `row` to `target_class`"
        vocab = getattr(self.dls, 'vocab', None)
        if is_listy(vocab): vocab = vocab[-1]  # dls.vocab can be listy (x+y vocabs); the targets are the last
        if isinstance(target_class, (int, np.integer)): target_idx = int(target_class)
        elif vocab is not None and target_class in getattr(vocab, 'o2i', {}): target_idx = int(vocab.o2i[target_class])
        elif vocab is not None and target_class in list(vocab): target_idx = list(vocab).index(target_class)
        else: raise ValueError(f"Could not resolve target_class={target_class!r} to a class index; expected an int or one of {list(vocab) if vocab is not None else '<no vocab>'}")
        if vocab is not None and not (0 <= target_idx < len(vocab)):
            raise ValueError(f"target_class index {target_idx} out of range for vocab of size {len(vocab)}")
        def _pred_idx(p): return int(array(p).argmax())  # the model's predicted class index for prob vector `p`
        cur = row.copy()
        _,_,probs = self.predict(cur)
        best_prob = float(probs[target_idx])
        if _pred_idx(probs) == target_idx:
            return SimpleNamespace(row=cur, changes={}, success=True, target_prob=best_prob, iterations=0)
        cont_names, cat_names = list(self.dls.cont_names), list(self.dls.cat_names)
        # Scale continuous nudges by each feature's RAW std, which `Normalize.setups` stored on the proc
        # (do NOT read train_ds.conts.std(): that frame is already normalized to ~unit std in place).
        stds = {}
        norm_proc = getattr(self.dls.procs, 'normalize', None)
        if norm_proc is None:
            for p in (self.dls.procs or []):
                if hasattr(p, 'stds') and isinstance(getattr(p, 'stds'), dict): norm_proc = p; break
        if norm_proc is not None:
            try: stds = dict(getattr(norm_proc, 'stds', {}) or {})
            except Exception: stds = {}
        cat_levels = {}
        cat_proc = getattr(self.dls.procs, 'categorify', None)
        for n in cat_names:
            levels = None
            if cat_proc is not None and n in getattr(cat_proc, 'classes', {}):
                levels = [l for l in list(cat_proc.classes[n]) if l != '#na#']
            if not levels:
                try: levels = list(self.dls.train_ds.items[n].unique())
                except Exception: levels = []
            cat_levels[n] = levels
        changes = {}
        iters = 0
        while iters < max_iter:
            iters += 1
            best = None  # (gain, feature, new_value)
            for n in cont_names:
                old = cur[n]
                if not np.isfinite(float(old)): continue
                step = cont_step * (float(stds.get(n, 1.0)) or 1.0)
                for new in (float(old)+step, float(old)-step):
                    trial = cur.copy(); trial[n] = new
                    _,_,p = self.predict(trial)
                    gain = float(p[target_idx]) - best_prob
                    if best is None or gain > best[0]: best = (gain, n, new)
            for n in cat_names:
                old = cur[n]
                for new in cat_levels.get(n, []):
                    if new == old: continue
                    trial = cur.copy(); trial[n] = new
                    _,_,p = self.predict(trial)
                    gain = float(p[target_idx]) - best_prob
                    if best is None or gain > best[0]: best = (gain, n, new)
            if best is None or best[0] <= 0: break
            _, feat, new_val = best
            if feat not in changes: changes[feat] = (cur[feat], new_val)
            else: changes[feat] = (changes[feat][0], new_val)
            cur[feat] = new_val
            _,_,probs = self.predict(cur)
            best_prob = float(probs[target_idx])
            if _pred_idx(probs) == target_idx:
                return SimpleNamespace(row=cur, changes=changes, success=True, target_prob=best_prob, iterations=iters)
        return SimpleNamespace(row=cur, changes=changes, success=False, target_prob=best_prob, iterations=iters)

In [ ]:
show_doc(TabularLearner, title_level=3)

<h3 id="TabularLearner" class="doc_header"><code>class</code> <code>TabularLearner</code><a href="" class="source_link" style="float:right">[source]</a></h3>

> <code>TabularLearner</code>(**`dls`**, **`model`**, **`loss_func`**=*`None`*, **`opt_func`**=*`Adam`*, **`lr`**=*`0.001`*, **`splitter`**=*`trainable_params`*, **`cbs`**=*`None`*, **`metrics`**=*`None`*, **`path`**=*`None`*, **`model_dir`**=*`'models'`*, **`wd`**=*`None`*, **`wd_bn_bias`**=*`False`*, **`train_bn`**=*`True`*, **`moms`**=*`(0.95, 0.85, 0.95)`*) :: `Learner`

`Learner` for tabular data

It works exactly as a normal `Learner`, the only difference is that it implements a `predict` method specific to work on a row of data.

In [ ]:
#|export
@delegates(Learner.__init__)
def tabular_learner(
        dls:TabularDataLoaders,
        layers:list=None, # Size of the layers generated by `LinBnDrop`
        emb_szs:list=None, # Tuples of `n_unique, embedding_size` for all categorical features
        config:dict=None, # Config params for TabularModel from `tabular_config`
        n_out:int=None, # Final output size of the model
        y_range:Tuple=None, # Low and high for the final sigmoid function
        **kwargs
):
    "Get a `Learner` using `dls`, with `metrics`, including a `TabularModel` created using the remaining params."
    if config is None: config = tabular_config()
    if layers is None: layers = [200,100]
    to = dls.train_ds
    emb_szs = get_emb_sz(dls.train_ds, {} if emb_szs is None else emb_szs)
    if n_out is None: n_out = get_c(dls)
    assert n_out, "`n_out` is not defined, and could not be inferred from data, set `dls.c` or pass `n_out`"
    if y_range is None and 'y_range' in config: y_range = config.pop('y_range')
    model = TabularModel(emb_szs, len(dls.cont_names), n_out, layers, y_range=y_range, **config)
    return TabularLearner(dls, model, **kwargs)

If your data was built with fastai, you probably won't need to pass anything to `emb_szs` unless you want to change the default of the library (produced by `get_emb_sz`), same for `n_out` which should be automatically inferred. `layers` will default to `[200,100]` and is passed to `TabularModel` along with the `config`.

Use `tabular_config` to create a `config` and customize the model used. There is just easy access to `y_range` because this argument is often used.

All the other arguments are passed to `Learner`.

In [ ]:
path = untar_data(URLs.ADULT_SAMPLE)
df = pd.read_csv(path/'adult.csv')
cat_names = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race']
cont_names = ['age', 'fnlwgt', 'education-num']
procs = [Categorify, FillMissing, Normalize]
dls = TabularDataLoaders.from_df(df, path, procs=procs, cat_names=cat_names, cont_names=cont_names, 
                                 y_names="salary", valid_idx=list(range(800,1000)), bs=64)
learn = tabular_learner(dls)

In [ ]:
show_doc(TabularLearner.predict)

<h4 id="TabularLearner.predict" class="doc_header"><code>TabularLearner.predict</code><a href="__main__.py#L5" class="source_link" style="float:right">[source]</a></h4>

> <code>TabularLearner.predict</code>(**`row`**)

Predict on a Pandas Series

We can pass in an individual row of data into our `TabularLearner`'s `predict` method. It's output is slightly different from the other `predict` methods, as this one will always return the input as well:

In [ ]:
row, clas, probs = learn.predict(df.iloc[0])

In [ ]:
row.show()

,workclass,education,marital-status,occupation,relationship,race,education-num_na,age,fnlwgt,education-num,salary
0,Private,Assoc-acdm,Married-civ-spouse,#na#,Wife,White,False,49.0,101320.001685,12.0,<50k


In [ ]:
clas, probs

(tensor(0), tensor([0.5264, 0.4736]))

The `counterfactual` method helps you understand a model's decision boundary. Given a `row` and a desired `target_class`, it greedily perturbs one feature at a time (nudging continuous features by `cont_step` scaled by their std, and swapping categorical features to other observed levels) until the prediction flips to `target_class` or `max_iter` is reached. It returns a `SimpleNamespace` with the perturbed `row`, a `changes` dict of `{feature: (old, new)}`, a `success` flag, the achieved `target_prob`, and the number of `iterations` used:

In [ ]:
show_doc(TabularLearner.counterfactual)

In [ ]:
#|hide
# counterfactual: hermetic synthetic test (no untar_data/network, no GPU).
# `grp` perfectly determines the label, so swapping it is a guaranteed, learnable flip.
import pandas as pd
_tdf = pd.DataFrame({
    'a': [float(i%10) for i in range(400)],
    'b': [float((i*3)%7) for i in range(400)],
    'grp': ['x' if i%2==0 else 'y' for i in range(400)],
})
_tdf['label'] = (_tdf['grp']=='y').astype(int)  # label is a pure function of `grp`
_dls = TabularDataLoaders.from_df(_tdf, procs=[Categorify, Normalize], cat_names=['grp'],
                                  cont_names=['a','b'], y_names='label',
                                  valid_idx=list(range(300,400)), bs=32)
_learn = tabular_learner(_dls, layers=[20,10])
_learn.fit_one_cycle(6)
_r = _tdf.iloc[0].drop('label')              # a grp=='x' row -> predicts class 0
# Work in integer class indices from the probs vector (predict's decoded class is a tensor,
# not a vocab label string, so never stringify it or compare against vocab labels).
_cur_idx = int(_learn.predict(_r)[2].argmax())
_target_idx = 1 - _cur_idx                    # 2-class synthetic setup: flip target is the complement

# 1) a genuinely achievable flip MUST succeed and the re-predicted class MUST equal the target (not gated).
_cf = _learn.counterfactual(_r, target_class=_target_idx, max_iter=50)
assert set(vars(_cf).keys()) == {'row','changes','success','target_prob','iterations'}
assert isinstance(_cf.changes, dict)
assert _cf.success, f"expected a flip to class {_target_idx} to succeed, got {_cf}"
assert int(_learn.predict(_cf.row)[2].argmax()) == _target_idx

# 2) already-at-target is a no-op: 0 iterations, empty changes, success.
_noop = _learn.counterfactual(_r, target_class=_cur_idx, max_iter=50)
test_eq(_noop.iterations, 0)
test_eq(_noop.changes, {})
assert _noop.success

# 3) error paths: unresolvable label and out-of-range int index both raise ValueError.
test_fail(lambda: _learn.counterfactual(_r, target_class='not_a_class', max_iter=1))
test_fail(lambda: _learn.counterfactual(_r, target_class=999, max_iter=1))

In [ ]:
#|hide
#test y_range is passed
learn = tabular_learner(dls, y_range=(0,32))
assert isinstance(learn.model.layers[-1], SigmoidRange)
test_eq(learn.model.layers[-1].low, 0)
test_eq(learn.model.layers[-1].high, 32)

learn = tabular_learner(dls, config = tabular_config(y_range=(0,32)))
assert isinstance(learn.model.layers[-1], SigmoidRange)
test_eq(learn.model.layers[-1].low, 0)
test_eq(learn.model.layers[-1].high, 32)

In [ ]:
#|export
@typedispatch
def show_results(x:Tabular, y:Tabular, samples, outs, ctxs=None, max_n=10, **kwargs):
    df = x.all_cols[:max_n]
    for n in x.y_names: df[n+'_pred'] = y[n][:max_n].values
    display_df(df)

## Export -

In [ ]:
#|hide
from nbdev import nbdev_export
nbdev_export()

Converted 00_torch_core.ipynb.
Converted 01_layers.ipynb.
Converted 02_data.load.ipynb.
Converted 03_data.core.ipynb.
Converted 04_data.external.ipynb.
Converted 05_data.transforms.ipynb.
Converted 06_data.block.ipynb.
Converted 07_vision.core.ipynb.
Converted 08_vision.data.ipynb.
Converted 09_vision.augment.ipynb.
Converted 09b_vision.utils.ipynb.
Converted 09c_vision.widgets.ipynb.
Converted 10_tutorial.pets.ipynb.
Converted 11_vision.models.xresnet.ipynb.
Converted 12_optimizer.ipynb.
Converted 13_callback.core.ipynb.
Converted 13a_learner.ipynb.
Converted 13b_metrics.ipynb.
Converted 14_callback.schedule.ipynb.
Converted 14a_callback.data.ipynb.
Converted 15_callback.hook.ipynb.
Converted 15a_vision.models.unet.ipynb.
Converted 16_callback.progress.ipynb.
Converted 17_callback.tracker.ipynb.
Converted 18_callback.fp16.ipynb.
Converted 18a_callback.training.ipynb.
Converted 19_callback.mixup.ipynb.
Converted 20_interpret.ipynb.
Converted 20a_distributed.ipynb.
Converted 21_vision.l